In [1]:
import pendulum

start_date = pendulum.date(2018, 1, 1)
end_date = pendulum.date(2024, 1, 1)

i = 0
deltas = []
for i in range(100):
    dt = start_date.add(months=i)
    if dt > end_date:
        break
    deltas.append((str(dt.year), str(dt.month) if dt.month > 9 else f"0{dt.month}"))

deltas

ModuleNotFoundError: No module named 'pendulum'

In [12]:
urls = [f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{t[0]}-{t[1]}.parquet" for t in deltas]

In [15]:
import httpx
import pathlib as plb

_dir = plb.Path('<REDACTED_HOST_HOME>/Documents/code_projects/local-dwh/tmp') / 'data'

for url in urls:
    fn = url.split("/")[-1]
    print(f"Downloading {url} to {_dir}/{fn}")
    r = httpx.get(url, timeout=120)
    with open(f"{_dir}/{fn}", "wb") as f:
        f.write(r.content)

In [1]:
import hvac
import os

client = hvac.Client(
    url="http://localstack.local:8200",
    token=os.getenv("VAULT_TOKEN")
)

In [2]:
read_response = client.secrets.kv.read_secret_version(path='default/minio/localstack')

/tmp/ipykernel_126869/1800919111.py:1: DeprecationWarning: The raise_on_deleted_version parameter will change its default value to False in hvac v3.0.0. The current default of True will preserve previous behavior. To use the old behavior with no warning, explicitly set this value to True. See https://github.com/hvac/hvac/pull/907
  read_response = client.secrets.kv.read_secret_version(path='default/minio/localstack')


In [3]:
access_key_id, secret_access_key = read_response['data']['data']['access_key'], read_response['data']['data']['secret_key']

In [4]:
secret_access_key

'<REDACTED_MINIO_SECRET_ACCESS_KEY>'

In [5]:
import duckdb
import os

con = duckdb.connect(database=':memory:')

con.execute(
    f"""
    CREATE OR REPLACE SECRET secret (
        TYPE s3,
        PROVIDER config,
        KEY_ID 'minio',
        SECRET '{secret_access_key}',
        REGION 'us-east-1',
        ENDPOINT 'orangepi4a.local:9000',
        URL_STYLE 'path',
        USE_SSL 'false'
    );
    """
)

In [6]:
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("INSTALL ducklake;")
con.execute("LOAD ducklake;")

# con.execute("SET s3_endpoint='orangepi4a.local:9000';")
# con.execute("SET s3_url_style='path';")
# con.execute("SET s3_access_key_id='minio';")
# con.execute("SET s3_secret_access_key='<REDACTED_MINIO_ROOT_PASSWORD>';")
# con.execute("SET s3_region='us-east-1';")
# con.execute("SET s3_use_ssl='false';")

In [11]:
conn_string = f"ducklake:postgres:dbname=localstack host={os.environ['PGHOST']} user={os.environ['PGUSER']} password=<REDACTED_POSTGRES_PASSWORD>"

In [12]:
# Override data path so that the location of the data doesn't have to match the location of the metadata
con.execute(f"ATTACH '{conn_string}' AS ducklake (DATA_PATH 's3://datalake', OVERRIDE_DATA_PATH true);")
con.execute("USE ducklake;")

In [13]:
con.execute("DROP TABLE IF EXISTS nyt_data;")
con.commit()

In [14]:
import pandas as pd

df = pd.read_parquet('data/yellow_tripdata_2024-01.parquet')

schema = con.execute("DESCRIBE SELECT * FROM read_parquet('data/yellow_tripdata_2024-01.parquet');").df()

schema

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [15]:
cols = []

for col in schema.to_dict(orient="records"):
    col = f'{col["column_name"]} {col["column_type"]}'
    cols.append(col)

create_stmt = f'CREATE TABLE nyt_data ({", ".join(cols)});'

partition_stmt = "ALTER TABLE nyt_data SET PARTITIONED BY (year(tpep_pickup_datetime), month(tpep_pickup_datetime));"

In [16]:
con.execute(create_stmt)

In [17]:
con.execute(partition_stmt)

In [18]:
con.commit()

In [20]:
con.execute("""
    INSERT INTO nyt_data 
    (
        SELECT * FROM read_parquet('data/*.parquet')
    );
""")

In [23]:
con.execute("SET s3_url_style='virtual';") 

In [ ]:
tst = """
    COPY (SELECT * FROM read_parquet('/home/vscode/workspace/applications/data_lake/data/yellow_tripdata_2018-01.parquet')) TO 's3://datalake/test.parquet' (FORMAT PARQUET);
"""

con.execute(tst)    

In [19]:
stmt = """
   SELECT * FROM nyt_data LIMIT 1;
"""

In [20]:
con.execute(stmt).df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee


In [10]:
con.commit()

In [11]:
con.close()

In [ ]:
#con.execute("DROP TABLE nyt_data;")

In [18]:
con.execute("LIST TABLES;")

ParserException: Parser Error: syntax error at or near "LIST"

LINE 1: LIST TABLES;
        ^